# Разведочный анализ данных виртуального датчика

## 1. Загрузка и первичный осмотр данных

**Цель исследования** — построить виртуальный датчик, который оценивает лабораторный показатель `Lab` по истории технологических измерений.

На первом этапе необходимо проверить структуру и качество исходных данных:

- типы и содержимое колонок;
- временной диапазон и частоту измерений;
- пропуски и значения `Bad` и `Failed`;
- дубликаты строк и временных меток;
- возможность связывания технологических и лабораторных измерений по `Date`.

Используются два файла:

- `data_2.csv` — регулярные технологические измерения;
- `lab_data_2.csv` — моменты измерения и значения лабораторного показателя `Lab`.

При загрузке технологических данных строки `Bad` и `Failed` заменяются на `NaN`. Даты преобразуются в `datetime` по явно заданному формату, чтобы исключить неоднозначную интерпретацию дня и месяца.

In [1]:
import pandas as pd
pd.options.display.max_rows = 50
pd.options.display.float_format = "{:.5f}".format

In [2]:
df = pd.read_csv('../data/raw/data_2.csv', sep=';', na_values=['Bad', 'Failed'])
df['Date'] = pd.to_datetime(df['Date'], format='%d.%m.%Y %H:%M')
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 196315 entries, 0 to 196314
Data columns (total 76 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Date             196315 non-null  datetime64[us]
 1   KGPN:13-FIC3147  196311 non-null  float64       
 2   KGPN:13-TIC1324  196283 non-null  float64       
 3   KGPN:13-TIC1309  196272 non-null  float64       
 4   KGPN:13-TIC1308  196257 non-null  float64       
 5   KGPN:13-TIC1293  196315 non-null  float64       
 6   KGPN:13-TI1330   196279 non-null  float64       
 7   KGPN:13-TI1328   196274 non-null  float64       
 8   KGPN:13-TI1307   196315 non-null  float64       
 9   KGPN:13-TI1306   196305 non-null  float64       
 10  KGPN:13-TI1147   196315 non-null  float64       
 11  KGPN:13-PI2190   194598 non-null  float64       
 12  KGPN:13-FIC3188  196315 non-null  float64       
 13  KGPN:13-FIC3149  196314 non-null  float64       
 14  KGPN:13-FIC3143  196315 non-nul

,Date,KGPN:13-FIC3147,KGPN:13-TIC1324,KGPN:13-TIC1309,KGPN:13-TIC1308,KGPN:13-TIC1293,KGPN:13-TI1330,KGPN:13-TI1328,KGPN:13-TI1307,KGPN:13-TI1306,...,KGPN:12-TIC1196,KGPN:12-PI2107,KGPN:12-PI2106,KGPN:12-PI2105,KGPN:12-FIC3090,KGPN:12-TI1234,KGPN:13-TIC1291,KGPN:13-PIC2183,KGPN:13-PI2173,KGPN:13-PI2199
count,196315,196311.00000,196283.00000,196272.00000,196257.00000,196315.00000,196279.00000,196274.00000,196315.00000,196305.00000,...,193553.00000,196279.00000,195299.00000,196280.00000,196298.00000,196227.00000,196274.00000,195588.00000,196274.00000,196276.00000
mean,2023-03-17 15:30:00,6.38410,52.83865,121.08736,70.68812,164.11455,51.95334,57.56477,126.68631,122.08609,...,90.24519,1.22882,1.01790,1.23816,4.93562,225.93650,64.89942,0.87372,1.12096,1.11448
min,2021-05-05 00:00:00,0.00000,-21.95110,-1.29302,-9.11160,3.59776,-12.13672,6.06341,7.06761,-2.63790,...,-17.08643,-0.01430,-0.00841,-0.00869,0.00000,-6.99119,-13.98412,-0.00063,-0.00024,-0.00059
25%,2022-04-10 19:45:00,5.00767,49.85116,123.87014,71.94610,158.94441,50.53743,56.10891,130.65449,122.57376,...,93.74718,1.28285,0.76311,1.29219,5.10890,223.00800,66.13840,0.92985,1.18593,1.17840
50%,2023-03-17 15:30:00,6.82884,56.65017,127.79742,74.34821,181.50157,54.90657,59.58022,134.57211,130.02328,...,97.44543,1.31102,1.32331,1.32001,5.31391,243.64284,68.78717,0.94000,1.19797,1.19052
75%,2024-02-21 11:15:00,8.43148,61.44699,130.66477,76.77368,187.61505,58.98785,62.84374,138.07175,134.57985,...,98.73840,1.34518,1.35665,1.35352,5.61683,251.55751,72.00773,0.95014,1.20561,1.19873
max,2025-01-27 07:00:00,42.03432,120.70780,193.01454,164.88953,228.35831,140.51729,181.14322,172.47337,158.57036,...,182.62642,1.92128,1.69133,1.70181,8.01711,316.33469,190.49736,1.31383,1.52981,1.51127
std,NaN,3.16295,15.76193,23.37614,15.15880,41.77314,13.70128,10.80179,27.11464,25.96317,...,22.88295,0.30218,0.55309,0.30116,1.47078,50.93859,17.14473,0.22283,0.25784,0.25520


In [3]:
lab_df = pd.read_csv('../data/raw/lab_data_2.csv', sep=';')
lab_df['Date'] = pd.to_datetime(lab_df['Date'], format='%d.%m.%Y %H:%M')
lab_df.info()
lab_df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 755 entries, 0 to 754
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    755 non-null    datetime64[us]
 1   Lab     755 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 11.9 KB


,Date,Lab
count,755,755.00000
mean,2022-11-25 08:52:12.715231,649.67060
min,2021-05-05 07:00:00,613.10000
25%,2021-11-13 19:00:00,645.70000
50%,2022-10-07 12:00:00,649.20000
75%,2023-10-31 07:00:00,653.10000
max,2025-01-27 07:00:00,797.90000
std,NaN,9.55875


In [4]:
df.dtypes

Date               datetime64[us]
KGPN:13-FIC3147           float64
KGPN:13-TIC1324           float64
KGPN:13-TIC1309           float64
KGPN:13-TIC1308           float64
                        ...      
KGPN:12-TI1234            float64
KGPN:13-TIC1291           float64
KGPN:13-PIC2183           float64
KGPN:13-PI2173            float64
KGPN:13-PI2199            float64
Length: 76, dtype: object

In [5]:
lab_df.dtypes

Date    datetime64[us]
Lab            float64
dtype: object

In [6]:
df.shape

(196315, 76)

In [7]:
lab_df.shape

(755, 2)

In [8]:
df['Date'].min()

Timestamp('2021-05-05 00:00:00')

In [9]:
df['Date'].max()

Timestamp('2025-01-27 07:00:00')

In [10]:
lab_df['Date'].min()

Timestamp('2021-05-05 07:00:00')

In [11]:
lab_df['Date'].max()

Timestamp('2025-01-27 07:00:00')

### Структура данных

Технологический набор содержит **196 315 строк и 76 колонок**: временную метку `Date` и 75 числовых датчиков. Период наблюдений — с **5 мая 2021 года** по **27 января 2025 года**.

Лабораторный набор содержит **755 наблюдений** и две колонки: `Date` и `Lab`. Пропусков в лабораторных значениях и временных метках нет.

Поэтому количество объектов для обучения определяется прежде всего числом лабораторных измерений, а не числом строк технологической таблицы. Каждая 10-минутная строка сама по себе не является размеченным примером, поскольку значение `Lab` известно только в 755 временных точках.

Описательная статистика показывает, что датчики имеют разные масштабы значений. Единицы измерения и физический смысл тегов в исходных таблицах не указаны, поэтому пока нельзя делать обоснованные выводы о параметрах. Разницу масштабов необходимо учитывать при обучении моделей.

In [12]:
df.nunique()

Date               196315
KGPN:13-FIC3147    177899
KGPN:13-TIC1324    193686
KGPN:13-TIC1309    182574
KGPN:13-TIC1308    184504
                    ...  
KGPN:12-TI1234     190807
KGPN:13-TIC1291    185280
KGPN:13-PIC2183    143725
KGPN:13-PI2173     160761
KGPN:13-PI2199     163826
Length: 76, dtype: int64

In [13]:
lab_df.nunique()

Date    755
Lab     241
dtype: int64

In [14]:
df.duplicated().sum()

np.int64(0)

In [15]:
df['Date'].duplicated().sum()

np.int64(0)

In [16]:
lab_df.duplicated().sum()

np.int64(0)

In [17]:
lab_df['Date'].duplicated().sum()

np.int64(0)

In [18]:
df.isnull().sum().sort_values(ascending=False)

KGPN:12-FIC3093    20563
KGPN:12-TIC1196     2762
KGPN:13-FIC3144     2162
KGPN:13-PI2191      1740
KGPN:13-PI2203      1730
                   ...  
KGPN:13-FIC3143        0
KGPN:12-PDI2123        0
KGPN:12-FIC3107        0
KGPN:12-FIC3106        0
KGPN:12-FI3091         0
Length: 76, dtype: int64

In [19]:
missing_summary_df = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": df.isnull().mean()*100
})

In [20]:
missing_summary_df.sort_values('missing_count', ascending=False)

,missing_count,missing_pct
KGPN:12-FIC3093,20563,10.47449
KGPN:12-TIC1196,2762,1.40692
KGPN:13-FIC3144,2162,1.10129
KGPN:13-PI2191,1740,0.88633
KGPN:13-PI2203,1730,0.88124
...,...,...
KGPN:13-FIC3143,0,0.00000
KGPN:12-PDI2123,0,0.00000
KGPN:12-FIC3107,0,0.00000
KGPN:12-FIC3106,0,0.00000


In [21]:
lab_df.isnull().sum().sort_values(ascending=False)

Date    0
Lab     0
dtype: int64

In [22]:
missing_summary_lab_df = pd.DataFrame({
			'missing_count': lab_df.isnull().sum(),
			'missing_pct': lab_df.isnull().mean()*100
	})

In [23]:
missing_summary_lab_df.sort_values('missing_count', ascending=False)

,missing_count,missing_pct
Date,0,0.00000
Lab,0,0.00000


In [24]:
df['Date'].is_monotonic_increasing

True

In [25]:
lab_df['Date'].is_monotonic_increasing

True

### Пропуски и дубликаты

Пропуски присутствуют не во всех технологических колонках, а их доля существенно различается между датчиками.

Наибольшее число пропусков наблюдается у `KGPN:12-FIC3093`: **20 563 значения**, или **10,47 %** всех наблюдений. У остальных наиболее проблемных датчиков доля пропусков значительно ниже: например, у `KGPN:12-TIC1196` она составляет около **1,41 %**.

Пропуски временных меток и пропуски значений датчиков представляют собой разные проблемы. Регулярная временная сетка не гарантирует, что в каждый момент времени доступны показания всех датчиков.

Полных дубликатов строк и дубликатов `Date` в обоих наборах не обнаружено, при этом значения `Lab` дубликатами не считаются: разные лабораторные измерения могут давать одинаковый результат, в том числе из-за округления.

На этапе построения признаков потребуется отдельно контролировать полноту данных внутри выбранных временных окон.

In [26]:
df['Date'].diff()

0                    NaT
1        0 days 00:10:00
2        0 days 00:10:00
3        0 days 00:10:00
4        0 days 00:10:00
               ...      
196310   0 days 00:10:00
196311   0 days 00:10:00
196312   0 days 00:10:00
196313   0 days 00:10:00
196314   0 days 00:10:00
Name: Date, Length: 196315, dtype: timedelta64[us]

In [27]:
lab_df['Date'].diff()

0                 NaT
1     0 days 10:30:00
2     0 days 06:30:00
3     0 days 03:00:00
4     0 days 04:00:00
            ...      
750   3 days 00:00:00
751   2 days 00:00:00
752   2 days 00:00:00
753   3 days 00:00:00
754   2 days 00:00:00
Name: Date, Length: 755, dtype: timedelta64[us]

In [28]:
lab_intervals_df = pd.DataFrame({
				'current_date': lab_df['Date'],
				'previous_date': lab_df['Date'].shift(1),
				'interval': lab_df['Date'].diff()
		})

In [29]:
lab_intervals_df.iloc[1:, 0:].sort_values('interval', ascending=True).head(5)

,current_date,previous_date,interval
13,2021-05-09 07:00:00,2021-05-09 04:00:00,0 days 03:00:00
162,2021-10-18 07:00:00,2021-10-18 04:00:00,0 days 03:00:00
3,2021-05-06 03:00:00,2021-05-06 00:00:00,0 days 03:00:00
280,2022-03-23 07:00:00,2022-03-23 04:00:00,0 days 03:00:00
54,2021-06-05 10:00:00,2021-06-05 07:00:00,0 days 03:00:00


In [30]:
lab_intervals_df.iloc[1:, 0:].sort_values('interval', ascending=False).head(5)

,current_date,previous_date,interval
645,2024-06-03 22:00:00,2024-04-29 07:00:00,35 days 15:00:00
109,2021-08-19 07:00:00,2021-08-01 07:00:00,18 days 00:00:00
276,2022-03-19 07:00:00,2022-03-05 07:00:00,14 days 00:00:00
588,2023-12-30 22:00:00,2023-12-18 07:00:00,12 days 15:00:00
111,2021-08-27 19:00:00,2021-08-20 07:00:00,7 days 12:00:00


In [31]:
df['Date'].diff().value_counts()

Date
0 days 00:10:00    196314
Name: count, dtype: int64

In [32]:
lab_df['Date'].diff().value_counts()

Date
2 days 00:00:00     253
1 days 00:00:00     246
3 days 00:00:00     109
0 days 03:00:00      10
0 days 16:00:00       9
                   ... 
5 days 11:00:00       1
35 days 15:00:00      1
6 days 03:00:00       1
1 days 14:30:00       1
0 days 15:00:00       1
Name: count, Length: 75, dtype: int64

### Временная структура измерений

Обе таблицы отсортированы по времени.

Технологические измерения образуют строго регулярную сетку: каждый из **196 314 интервалов** между соседними записями составляет ровно **10 минут**. Пропущенных временных меток и нерегулярных интервалов в технологическом файле не обнаружено.

Лабораторные измерения, напротив, выполняются нерегулярно. Наиболее частые интервалы — **2 дня** и **1 день**, однако встречаются как короткие промежутки в несколько часов, так и многодневные перерывы.

Минимальный зарегистрированный интервал составляет **3 часа**. Один из таких интервалов наблюдается между 04:00 и 07:00 9 мая 2021 года.

Максимальный перерыв составляет **35 дней 15 часов** — между 07:00 29 апреля и 22:00 3 июня 2024 года.

In [33]:
merge_df = df.merge(lab_df, on='Date', how='left', validate='one_to_one')

In [34]:
merge_df['Lab'].notnull().sum()

np.int64(755)

In [35]:
merge_df['Lab'].isnull().sum()

np.int64(195560)

In [36]:
merge_df.loc[merge_df['Lab'].notnull(), ('Date', 'Lab')]

,Date,Lab
42,2021-05-05 07:00:00,623.80000
105,2021-05-05 17:30:00,657.00000
144,2021-05-06 00:00:00,656.30000
162,2021-05-06 03:00:00,658.20000
186,2021-05-06 07:00:00,661.10000
...,...,...
195018,2025-01-18 07:00:00,653.50000
195306,2025-01-20 07:00:00,649.70000
195594,2025-01-22 07:00:00,646.80000
196026,2025-01-25 07:00:00,646.60000


### Связывание технологических и лабораторных данных

Таблицы объединены по точному совпадению временных меток `Date`. Параметр `validate='one_to_one'` дополнительно подтверждает, что каждая временная метка уникальна в обоих наборах и одна строка технологических данных соответствует не более чем одному лабораторному измерению.

Все **755 лабораторных измерений** имеют точное соответствие в технологической таблице.

Для последующего обучения необходимо использовать моменты лабораторных измерений как точки формирования прогноза. Технологические признаки должны рассчитываться только по данным, доступным не позже соответствующего момента времени.

In [37]:
raw_sensor_df = pd.read_csv('../data/raw/data_2.csv', sep=';', dtype='str', na_filter=False, keep_default_na=False)

In [38]:
raw_sensor_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 196315 entries, 0 to 196314
Data columns (total 76 columns):
 #   Column           Non-Null Count   Dtype
---  ------           --------------   -----
 0   Date             196315 non-null  str  
 1   KGPN:13-FIC3147  196315 non-null  str  
 2   KGPN:13-TIC1324  196315 non-null  str  
 3   KGPN:13-TIC1309  196315 non-null  str  
 4   KGPN:13-TIC1308  196315 non-null  str  
 5   KGPN:13-TIC1293  196315 non-null  str  
 6   KGPN:13-TI1330   196315 non-null  str  
 7   KGPN:13-TI1328   196315 non-null  str  
 8   KGPN:13-TI1307   196315 non-null  str  
 9   KGPN:13-TI1306   196315 non-null  str  
 10  KGPN:13-TI1147   196315 non-null  str  
 11  KGPN:13-PI2190   196315 non-null  str  
 12  KGPN:13-FIC3188  196315 non-null  str  
 13  KGPN:13-FIC3149  196315 non-null  str  
 14  KGPN:13-FIC3143  196315 non-null  str  
 15  KGPN:13-FIC3136  196315 non-null  str  
 16  KGPN:13-TI1334   196315 non-null  str  
 17  KGPN:13-PI2191   196315 non-null  str  


In [39]:
bad_result = raw_sensor_df.isin(['Bad']).sum().sort_values(ascending=False)
bad_result

KGPN:12-FIC3093    20563
KGPN:12-TIC1196     2762
KGPN:13-FIC3144     2162
KGPN:13-PI2191      1740
KGPN:13-PI2203      1730
                   ...  
KGPN:13-FIC3143        0
KGPN:12-PDI2123        0
KGPN:12-FIC3107        0
KGPN:12-FIC3106        0
KGPN:12-FI3091         0
Length: 76, dtype: int64

In [40]:
bad_result.sum()

np.int64(47609)

In [41]:
failed_result = raw_sensor_df.isin(['Failed']).sum().sort_values(ascending=False)
failed_result

KGPN:13-FIC3136    1
Date               0
KGPN:13-TIC1324    0
KGPN:13-FIC3147    0
KGPN:13-TIC1308    0
                  ..
KGPN:12-TI1234     0
KGPN:13-TIC1291    0
KGPN:13-PIC2183    0
KGPN:13-PI2173     0
KGPN:13-PI2199     0
Length: 76, dtype: int64

In [42]:
failed_result.sum()

np.int64(1)

In [43]:
empty_result = raw_sensor_df.eq('').sum()
empty_result

Date               0
KGPN:13-FIC3147    0
KGPN:13-TIC1324    0
KGPN:13-TIC1309    0
KGPN:13-TIC1308    0
                  ..
KGPN:12-TI1234     0
KGPN:13-TIC1291    0
KGPN:13-PIC2183    0
KGPN:13-PI2173     0
KGPN:13-PI2199     0
Length: 76, dtype: int64

In [44]:
empty_result.sum()

np.int64(0)

In [45]:
missing_counts = df.isnull().sum()
status_counts = raw_sensor_df.isin(['Bad', 'Failed']).sum()

(missing_counts == status_counts).all()

np.True_

### Происхождение пропусков

Для анализа происхождения пропусков технологический файл дополнительно загружен в исходном строковом представлении, без автоматического преобразования значений в числа и `NaN`.

В показаниях датчиков обнаружено **47 609 значений `Bad`** и **одно значение `Failed`**. Единственный статус `Failed` относится к датчику `KGPN:13-FIC3136`.

Общее количество статусов `Bad` и `Failed` составляет **47 610** и в точности совпадает с числом `NaN` в числовых колонках основного набора. Это совпадение выполняется как для всего набора, так и для каждого датчика отдельно. Следовательно, все пропуски технологических измерений явно представлены в исходном файле статусами `Bad` или `Failed`.

Таким образом, пропуски связаны с отмеченной в источнике недоступностью или некорректностью показаний.